# 📊 Ôn tập Metrics — MOTA, IDF1, HOTA

Notebook này giúp bạn:
1. Hiểu ý nghĩa và cách tính MOTA, IDF1, HOTA
2. Trả lời 3 câu True/False
3. Chạy YOLO trên một ảnh của video_1 để thấy detection (chưa có ID)
4. Đọc kết quả TrackEval sau khi chạy tracker

> **Kernel**: chọn `.venv` (Python 3.11) trong VSCode


## 1. Các metric quan trọng

### MOTA — Multi-Object Tracking Accuracy
$$MOTA = 1 - \frac{FN + FP + IDSW}{GT}$$

Trong đó:
- **FN** (False Negative): người bị bỏ sót
- **FP** (False Positive): hộp giả (detect ra vật không phải người)
- **IDSW** (ID Switch): số lần một người đổi ID
- **GT**: tổng số ground-truth detection

MOTA có thể âm! Ưu tiên detector chính xác hơn là tracker tốt.

---

### IDF1 — Identity F1 Score
$$IDF1 = \frac{2 \cdot IDTP}{2 \cdot IDTP + IDFP + IDFN}$$

- Đo khả năng **giữ đúng danh tính** xuyên suốt video
- Cao hơn MOTA khi tracker giữ ID ổn dù bỏ sót một số frame

---

### HOTA — Higher Order Tracking Accuracy
$$HOTA = \sqrt{DetA \times AssA}$$

- **DetA** (Detection Accuracy): chất lượng phát hiện (như AP)
- **AssA** (Association Accuracy): chất lượng liên kết danh tính
- HOTA **cân bằng** cả 2 khía cạnh → metric được khuyến nghị nhất hiện nay


## 2. Bảng so sánh nhanh

| Metric | Phản ánh điều gì | Ảnh hưởng bởi |
|--------|-----------------|---------------|
| MOTA   | Lỗi tổng thể (bỏ sót + giả + đổi ID) | Detector nhiều hơn tracker |
| IDF1   | Giữ đúng danh tính qua frame | Tracker (Re-ID) |
| HOTA   | Cân bằng detection + association | Cả hai |

### Khi nào dùng metric nào?
- So sánh **detector**: dùng MOTA
- So sánh **tracker** (cùng detector): dùng IDF1 hoặc HOTA
- **Báo cáo chính thức**: dùng HOTA


## 3. True/False — Kiểm tra hiểu biết (CP1)

Điền `True` hoặc `False` vào các ô sau:


In [ ]:
# ─── Điền câu trả lời của nhóm vào đây ───────────────────────────────────────

# Câu 1: MOTA có thể nhận giá trị âm (dưới 0).
cau_1 = True  # True: MOTA có thể âm khi (FN+FP+IDSW) > GT  # Điền True hoặc False

# Câu 2: IDF1 cao hơn nghĩa là tracker giữ ID ổn định hơn qua các frame.
cau_2 = True  # True: IDF1 đo F1-score của ID association  # Điền True hoặc False

# Câu 3: Tham số --iou trong run_tracking.py ảnh hưởng đến cách tracker
#         ghép ID giữa các frame.
cau_3 = False  # False: --iou là ngưỡng NMS của detector YOLO  # Điền True hoặc False

# ─────────────────────────────────────────────────────────────────────────────
# Chạy cell này để kiểm tra
answers = {"Câu 1": (cau_1, True), "Câu 2": (cau_2, True), "Câu 3": (cau_3, False)}

print("=" * 50)
print("Kết quả kiểm tra")
print("=" * 50)
all_correct = True
for q, (your_ans, correct) in answers.items():
    if your_ans is None:
        print(f"❓ {q}: Chưa điền")
        all_correct = False
    elif your_ans == correct:
        print(f"✅ {q}: {your_ans} — Đúng!")
    else:
        print(f"❌ {q}: {your_ans} — Sai. Đáp án: {correct}")
        all_correct = False

if all_correct:
    print("\n🎉 Tất cả đúng! Sẵn sàng sang CP2.")


## 4. Chạy YOLO trên một ảnh của video_1 (CP1)

Cell này hiển thị ảnh với bounding box người — chưa có ID, chỉ có detection.


In [ ]:
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ─── Cấu hình ─────────────────────────────────────────────────────────────────
LAB_DATA = os.environ.get("LAB_DATA", "")  # Set biến môi trường trước khi mở notebook

if not LAB_DATA:
    print("⚠️  Chưa đặt LAB_DATA. Hãy chạy:")
    print('   $env:LAB_DATA = "C:/path/to/lab_data"')
    print("   rồi mở lại notebook từ terminal đó.")
else:
    video1_img_dir = Path(LAB_DATA) / "video_1" / "img1"
    img_files = sorted(video1_img_dir.glob("*.jpg"))
    
    if not img_files:
        print(f"❌ Không tìm thấy ảnh trong: {video1_img_dir}")
    else:
        # Lấy frame đầu tiên
        sample_img_path = img_files[0]
        print(f"Đang detect trên: {sample_img_path.name}")


In [ ]:
from ultralytics import YOLO

# Load model (tự download yolo11n.pt lần đầu)
model = YOLO("yolo11n.pt")

# Detect trên ảnh mẫu — class 0 = person
results = model.predict(
    str(sample_img_path),
    conf=0.3,
    iou=0.5,
    classes=[0],
    verbose=False
)

# Đọc ảnh và vẽ bounding box
img_bgr = cv2.imread(str(sample_img_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.imshow(img_rgb)
ax.set_title(f"YOLO Detection — {sample_img_path.name}\n(Chỉ có bounding box, CHƯA có ID)", fontsize=13)
ax.axis("off")

boxes = results[0].boxes
if boxes is not None:
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = float(box.conf[0])
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor='red', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 3, f"person {conf:.2f}", color='red', fontsize=8)

    n_det = len(boxes)
    ax.set_xlabel(f"Phát hiện được {n_det} người | conf=0.3 | iou=0.5", fontsize=11)

plt.tight_layout()
plt.show()
print(f"\n✅ Phát hiện {len(boxes) if boxes is not None else 0} người trong frame đầu tiên.")
print("   → Tracker sẽ gán ID và theo dõi qua các frame tiếp theo.")


## 5. Đọc kết quả TrackEval của video_1

Sau khi chạy `scripts/run_tracking.py` đủ frame và `scripts/run_mot_challenge.py`, đọc kết quả tại đây.


In [ ]:
import pandas as pd
from pathlib import Path

# Tìm file kết quả TrackEval
# File thường ở: runs/nop_bai/pedestrian_detailed.csv hoặc pedestrian_summary.txt
result_dirs = list(Path("runs").glob("**/*.csv")) + list(Path("runs").glob("**/*summary*.txt"))

print("Files kết quả tìm thấy:")
for f in result_dirs:
    print(f"  - {f}")

if not result_dirs:
    print("\n⏳ Chưa có kết quả. Chạy tracking và eval trước:")
    print("   python scripts/run_tracking.py --source $LAB_DATA/video_1/img1 ...")


In [ ]:
# ─── Bang tong hop ket qua ca 5 video ────────────────────
results_table = [
    {"Video": "video_1", "Tracker": "bytetrack", "conf": 0.30, "iou": 0.50, "HOTA (%)": 33.53, "MOTA (%)": 18.33, "IDF1 (%)": 27.52, "Danh gia": "Dinh luong TrackEval (IDSW: 20)"},
    {"Video": "video_2", "Tracker": "deepocsort", "conf": 0.25, "iou": 0.50, "HOTA (%)": "-", "MOTA (%)": "-", "IDF1 (%)": "-", "Danh gia": "Quan sat: Re-ID giu ID rat tot khi dong duc"},
    {"Video": "video_3", "Tracker": "bytetrack", "conf": 0.25, "iou": 0.50, "HOTA (%)": "-", "MOTA (%)": "-", "IDF1 (%)": "-", "Danh gia": "Quan sat: Second-match giup bat nguoi nho, mo"},
    {"Video": "video_4", "Tracker": "ocsort", "conf": 0.40, "iou": 0.50, "HOTA (%)": "-", "MOTA (%)": "-", "IDF1 (%)": "-", "Danh gia": "Quan sat: conf=0.40 loc phan chieu guong kinh"},
    {"Video": "video_5", "Tracker": "botsort", "conf": 0.30, "iou": 0.50, "HOTA (%)": "-", "MOTA (%)": "-", "IDF1 (%)": "-", "Danh gia": "Quan sat: CMC bu rung lac camera tren xe bus"},
]
df = pd.DataFrame(results_table)
print(df.to_string(index=False))


## 6. Giả thuyết và quan sát (CP1)

Ghi lại ý kiến của nhóm trước khi chạy thực nghiệm:


In [ ]:
# ─── Điền giả thuyết của nhóm ─────────────────────────────────────────────────

hypotheses = {
    "video_1": "(Điền dự đoán của nhóm)",
    "video_2": "(Điền dự đoán của nhóm)",
    "video_3": "(Điền dự đoán của nhóm)",
    "video_4": "(Điền dự đoán của nhóm)",
    "video_5": "(Điền dự đoán của nhóm)",
}

print("Giả thuyết của nhóm:")
print("-" * 60)
for video, hyp in hypotheses.items():
    print(f"{video}: {hyp}")
print("-" * 60)
print("→ So sánh với kết quả quan sát video sau khi chạy tracking.")
